## Линейная регрессия с логарифмом цены

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import pickle

```
Загрузка датасета
```

In [3]:
df = pd.read_csv('data/drom_archive_cleaned_2018-2025.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data/drom_archive_cleaned_2018-2025.csv'

In [ ]:
df.info()

```
Создание колонки с логарифмом цены
```

In [ ]:
df['log_price'] = np.log(df['Цена'])

```
Преобразование категориальных признаков
```

In [ ]:
categorical = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Поколение', 'Рестайлинг', 'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Месяц объявления', 'Возраст авто']

In [ ]:
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical), ('num', 'passthrough', numerical)], remainder='drop')

```
Разделение данных на целевую переменную и признаки
```

In [ ]:
y = df['log_price'] # Целевая переменная теперь логарифм цены
X = df.drop(['Цена', 'log_price'], axis=1)

```
Разделение на обучающую и тестовую выборки (2018-2023, 2024-2025)
```

In [ ]:
X_train = X[X['Год объявления'] <= 2023]
X_test = X[X['Год объявления'] >= 2024]

y_train = y[X['Год объявления'] <= 2023]
y_test = y[X['Год объявления'] >= 2024]
joblib.dump((X_train, X_test, y_train, y_test), "data/data_split_log.pkl")
X_train, X_test, y_train, y_test = joblib.load("data/data_split_log.pkl")

In [ ]:
model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])

```
Обучение и сохранение модели
```

In [ ]:
model.fit(X_train, y_train)
joblib.dump(model, "models/lr_log_model.pkl")

```
Предсказание на тестовой выборке
```

In [ ]:
y_pred_log = model.predict(X_test)

In [ ]:
y_pred = np.exp(y_pred_log) # Обратное преобразование прогнозов

```
Оценка модели
```

In [ ]:
lr_mse = mean_squared_error(y_test, y_pred)
lr_rmse = np.sqrt(lr_mse)
lr_mae = mean_absolute_error(y_test, y_pred)
lr_r2 = r2_score(y_test, y_pred)

```
Вывод результатов
```

In [ ]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [lr_mse, lr_rmse, lr_mae, lr_r2]
})

```
Сохранение метрик в отдельный файл
```

In [ ]:
metrics = {
    "model_name": "Linear Regression Log",
    "MSE": lr_mse,
    "RMSE": lr_rmse,
    "MAE": lr_mae,
    "R2": lr_r2
}

In [ ]:
with open("metrics/lr_log_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)